# Movie Recommendation System

## Model Development

This notebook focuses on preparing the data, building recommendation models, and evaluating their performance.

In [2]:
# Import the libraries required for model development.

import pandas as pd
import numpy as np

In [3]:
# Load the datasets required for model development.

movies = pd.read_csv("../data/movies.csv")
users = pd.read_csv("../data/users.csv")
ratings = pd.read_csv("../data/ratings.csv")

print("Movies:", movies.shape)
print("Users:", users.shape)
print("Ratings:", ratings.shape)

Movies: (3883, 3)
Users: (6040, 5)
Ratings: (1000209, 4)


In [4]:
# Prepare the ratings data for model training.

model_data = ratings[["user_id", "movie_id", "rating"]].copy()

print(model_data.head())
print("\nShape:", model_data.shape)

   user_id  movie_id  rating
0        1      1193       5
1        1       661       3
2        1       914       3
3        1      3408       4
4        1      2355       5

Shape: (1000209, 3)


In [5]:
# Split the data into training and testing sets.

from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    model_data,
    test_size=0.2,
    random_state=42
)

print("Training set:", train_data.shape)
print("Testing set:", test_data.shape)

Training set: (800167, 3)
Testing set: (200042, 3)


In [6]:
# Check the rating distribution in the training and testing sets.

print("Training rating distribution:")
print(train_data["rating"].value_counts().sort_index())

print("\nTesting rating distribution:")
print(test_data["rating"].value_counts().sort_index())

Training rating distribution:
rating
1     44773
2     86194
3    209081
4    279323
5    180796
Name: count, dtype: int64

Testing rating distribution:
rating
1    11401
2    21363
3    52116
4    69648
5    45514
Name: count, dtype: int64


In [7]:
# Create the user-movie rating matrix for collaborative filtering.

user_movie_matrix = train_data.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print("Matrix shape:", user_movie_matrix.shape)

Matrix shape: (6040, 3683)


In [8]:
# Calculate the sparsity of the training interaction matrix.

total_cells = user_movie_matrix.shape[0] * user_movie_matrix.shape[1]
rated_cells = user_movie_matrix.count().sum()

train_sparsity = 1 - (rated_cells / total_cells)

print("Total possible ratings:", total_cells)
print("Actual ratings:", rated_cells)
print("Training sparsity:", train_sparsity)

Total possible ratings: 22245320
Actual ratings: 800167
Training sparsity: 0.9640298723506787


In [9]:
# Prepare the user-movie matrix for calculating user similarity.

from sklearn.metrics.pairwise import cosine_similarity

user_movie_matrix_filled = user_movie_matrix.fillna(0)

print("Matrix shape:", user_movie_matrix_filled.shape)

Matrix shape: (6040, 3683)


In [10]:
# Calculate the similarity between users based on their movie ratings.

user_similarity = cosine_similarity(user_movie_matrix_filled)

print("User similarity matrix shape:", user_similarity.shape)

User similarity matrix shape: (6040, 6040)


In [11]:
# Find the most similar users to a selected user.

user_id = 1

user_index = user_movie_matrix.index.get_loc(user_id)

similar_users = user_similarity[user_index]

similar_user_indices = similar_users.argsort()[::-1][1:11]

similar_user_ids = user_movie_matrix.index[similar_user_indices]

similar_user_scores = similar_users[similar_user_indices]

similar_users_df = pd.DataFrame({
    "user_id": similar_user_ids,
    "similarity": similar_user_scores
})

similar_users_df

,user_id,similarity
0,5705,0.379568
1,5190,0.375027
2,5343,0.374112
3,1850,0.367116
4,1283,0.354903
5,184,0.349443
6,1476,0.336817
7,1481,0.326319
8,96,0.322746
9,5063,0.319972


In [12]:
# Generate movie recommendations based on the ratings of similar users.

similar_user_ids = similar_users_df["user_id"].tolist()

similar_users_ratings = train_data[
    train_data["user_id"].isin(similar_user_ids)
]

user_rated_movies = set(
    train_data[train_data["user_id"] == user_id]["movie_id"]
)

candidate_movies = similar_users_ratings[
    ~similar_users_ratings["movie_id"].isin(user_rated_movies)
]

print("Candidate movie ratings:", len(candidate_movies))

Candidate movie ratings: 494


In [13]:
# Rank candidate movies using the ratings and similarity of similar users.

similarity_map = dict(
    zip(
        similar_users_df["user_id"],
        similar_users_df["similarity"]
    )
)

candidate_movies = candidate_movies.copy()

candidate_movies["similarity"] = candidate_movies["user_id"].map(
    similarity_map
)

candidate_movies["weighted_rating"] = (
    candidate_movies["rating"] *
    candidate_movies["similarity"]
)

recommendation_scores = (
    candidate_movies
    .groupby("movie_id")
    .agg(
        score=("weighted_rating", "sum"),
        rating_count=("rating", "count")
    )
    .reset_index()
    .sort_values("score", ascending=False)
)

recommendation_scores.head(10)

,movie_id,score,rating_count
215,2080,9.314695,6
216,2081,9.265579,6
14,110,7.479896,5
203,2018,6.769531,5
221,2096,6.616689,5
35,364,6.331587,4
279,2987,6.145145,5
167,1580,6.026778,4
214,2078,6.012716,4
131,1225,5.943341,4


In [14]:
# Display the recommended movies with their titles.

recommendations = recommendation_scores.head(10).merge(
    movies[["movie_id", "title", "genres"]],
    on="movie_id",
    how="left"
)

recommendations[
    ["movie_id", "title", "genres", "score", "rating_count"]
]

,movie_id,title,genres,score,rating_count
0,2080,Lady and the Tramp (1955),Animation|Children's|Comedy|Musical|Romance,9.314695,6
1,2081,"Little Mermaid, The (1989)",Animation|Children's|Comedy|Musical|Romance,9.265579,6
2,110,Braveheart (1995),Action|Drama|War,7.479896,5
3,2018,Bambi (1942),Animation|Children's,6.769531,5
4,2096,Sleeping Beauty (1959),Animation|Children's|Musical,6.616689,5
5,364,"Lion King, The (1994)",Animation|Children's|Musical,6.331587,4
6,2987,Who Framed Roger Rabbit? (1988),Adventure|Animation|Film-Noir,6.145145,5
7,1580,Men in Black (1997),Action|Adventure|Comedy|Sci-Fi,6.026778,4
8,2078,"Jungle Book, The (1967)",Animation|Children's|Comedy|Musical,6.012716,4
9,1225,Amadeus (1984),Drama,5.943341,4


In [15]:
# Calculate a normalized recommendation score for each candidate movie.

recommendation_scores = (
    candidate_movies
    .groupby("movie_id")
    .agg(
        weighted_sum=("weighted_rating", "sum"),
        total_similarity=("similarity", "sum"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

recommendation_scores["score"] = (
    recommendation_scores["weighted_sum"] /
    recommendation_scores["total_similarity"]
)

recommendation_scores = recommendation_scores.sort_values(
    "score",
    ascending=False
)

recommendation_scores.head(10)

,movie_id,weighted_sum,total_similarity,rating_count,score
120,1198,3.378814,0.675763,2,5.0
11,62,1.599862,0.319972,1,5.0
17,161,1.613730,0.322746,1,5.0
346,3948,1.599862,0.319972,1,5.0
189,1848,1.747217,0.349443,1,5.0
248,2502,1.897842,0.379568,1,5.0
245,2463,1.747217,0.349443,1,5.0
281,2996,1.599862,0.319972,1,5.0
212,2067,1.631597,0.326319,1,5.0
210,2058,1.897842,0.379568,1,5.0


In [16]:
# Display the top recommendations with movie details.

recommendations = recommendation_scores.head(10).merge(
    movies[["movie_id", "title", "genres"]],
    on="movie_id",
    how="left"
)

recommendations[
    ["movie_id", "title", "genres", "score", "rating_count"]
]

,movie_id,title,genres,score,rating_count
0,1198,Raiders of the Lost Ark (1981),Action|Adventure,5.0,2
1,62,Mr. Holland's Opus (1995),Drama,5.0,1
2,161,Crimson Tide (1995),Drama|Thriller|War,5.0,1
3,3948,Meet the Parents (2000),Comedy,5.0,1
4,1848,"Borrowers, The (1997)",Adventure|Children's|Comedy|Fantasy,5.0,1
5,2502,Office Space (1999),Comedy|Romance,5.0,1
6,2463,Ruthless People (1986),Comedy,5.0,1
7,2996,Music of the Heart (1999),Drama,5.0,1
8,2067,Doctor Zhivago (1965),Drama|Romance|War,5.0,1
9,2058,"Negotiator, The (1998)",Action|Thriller,5.0,1


In [17]:
# Keep recommendations supported by at least five similar-user ratings.

reliable_recommendations = recommendation_scores[
    recommendation_scores["rating_count"] >= 5
].copy()

reliable_recommendations = reliable_recommendations.sort_values(
    "score",
    ascending=False
)

print("Reliable candidate movies:", len(reliable_recommendations))

Reliable candidate movies: 6


In [18]:
# Display the top reliable movie recommendations.

final_recommendations = reliable_recommendations.head(10).merge(
    movies[["movie_id", "title", "genres"]],
    on="movie_id",
    how="left"
)

final_recommendations[
    ["movie_id", "title", "genres", "score", "rating_count"]
]

,movie_id,title,genres,score,rating_count
0,2080,Lady and the Tramp (1955),Animation|Children's|Comedy|Musical|Romance,4.491651,6
1,110,Braveheart (1995),Action|Drama|War,4.423210,5
2,2081,"Little Mermaid, The (1989)",Animation|Children's|Comedy|Musical|Romance,4.327108,6
3,2018,Bambi (1942),Animation|Children's,3.766136,5
4,2096,Sleeping Beauty (1959),Animation|Children's|Musical,3.761056,5
5,2987,Who Framed Roger Rabbit? (1988),Adventure|Animation|Film-Noir,3.552533,5


In [19]:
# Prepare the user similarity matrix for rating prediction.

from sklearn.metrics import mean_squared_error

user_indices = {
    user_id: index
    for index, user_id in enumerate(user_movie_matrix.index)
}

print("Number of users:", len(user_indices))

Number of users: 6040


In [20]:
# Predict a user's rating for a movie using ratings from similar users.

def predict_rating(user_id, movie_id, k=10):
    
    if user_id not in user_indices:
        return np.nan
    
    user_index = user_indices[user_id]
    
    if movie_id not in user_movie_matrix.columns:
        return np.nan
    
    movie_ratings = user_movie_matrix[movie_id]
    
    rated_users = movie_ratings.notna()
    
    if not rated_users.any():
        return np.nan
    
    rated_user_indices = [
        user_indices[u]
        for u in movie_ratings[rated_users].index
    ]
    
    similarities = user_similarity[user_index][rated_user_indices]
    ratings_for_movie = movie_ratings[rated_users].values
    
    if len(similarities) == 0:
        return np.nan
    
    top_indices = np.argsort(similarities)[::-1][:k]
    
    top_similarities = similarities[top_indices]
    top_ratings = ratings_for_movie[top_indices]
    
    similarity_sum = np.abs(top_similarities).sum()
    
    if similarity_sum == 0:
        return np.nan
    
    predicted_rating = (
        np.sum(top_similarities * top_ratings) /
        similarity_sum
    )
    
    return predicted_rating

In [21]:
# Test the rating prediction function on one unseen rating.

sample = test_data.iloc[0]

prediction = predict_rating(
    sample["user_id"],
    sample["movie_id"]
)

print("User ID:", sample["user_id"])
print("Movie ID:", sample["movie_id"])
print("Actual rating:", sample["rating"])
print("Predicted rating:", prediction)

User ID: 5412
Movie ID: 2683
Actual rating: 2
Predicted rating: 2.9967637841213763


In [22]:
# Generate predictions for a sample of unseen test ratings.

evaluation_sample = test_data.sample(
    n=5000,
    random_state=42
)

predictions = []
actual_ratings = []

for _, row in evaluation_sample.iterrows():
    
    prediction = predict_rating(
        row["user_id"],
        row["movie_id"]
    )
    
    if not np.isnan(prediction):
        predictions.append(prediction)
        actual_ratings.append(row["rating"])

print("Predictions generated:", len(predictions))

Predictions generated: 5000


In [23]:
# Calculate the RMSE of the recommendation model.

rmse = np.sqrt(
    mean_squared_error(
        actual_ratings,
        predictions
    )
)

print("RMSE:", rmse)

RMSE: 0.9835396241235608


In [24]:
# Calculate the baseline RMSE using the average training rating.

from sklearn.metrics import mean_squared_error

baseline_rating = train_data["rating"].mean()

baseline_predictions = np.full(
    len(actual_ratings),
    baseline_rating
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        actual_ratings,
        baseline_predictions
    )
)

print("Baseline average rating:", baseline_rating)
print("Baseline RMSE:", baseline_rmse)

Baseline average rating: 3.5813473937315585
Baseline RMSE: 1.117356683608699


In [25]:
# Compare the baseline model with the collaborative filtering model.

comparison = pd.DataFrame({
    "Model": [
        "Baseline - Average Rating",
        "User-Based Collaborative Filtering"
    ],
    "RMSE": [
        baseline_rmse,
        rmse
    ]
})

comparison

,Model,RMSE
0,Baseline - Average Rating,1.117357
1,User-Based Collaborative Filtering,0.983540


In [26]:
# Calculate each user's average rating from the training data.

user_mean_ratings = train_data.groupby("user_id")["rating"].mean()

print(user_mean_ratings.head())

user_id
1    4.146341
2    3.733945
3    3.853659
4    4.210526
5    3.104575
Name: rating, dtype: float64


In [27]:
# Center each user's ratings around their personal average rating.

mean_centered_matrix = user_movie_matrix.sub(
    user_mean_ratings,
    axis=0
)

print(mean_centered_matrix.head())

movie_id      1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                                   ...   
1         0.853659   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
2              NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3              NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4              NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5              NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

movie_id  3943  3944  3945  3946  3947  3948  3949  3950  3951  3952  
user_id                                                               
1          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
5     

In [28]:
# Calculate user similarity using mean-centered ratings.

mean_centered_filled = mean_centered_matrix.fillna(0)

centered_user_similarity = cosine_similarity(
    mean_centered_filled
)

print("Similarity matrix shape:", centered_user_similarity.shape)

Similarity matrix shape: (6040, 6040)


In [29]:
# Predict ratings using mean-centered user-based collaborative filtering.

def predict_centered_rating(user_id, movie_id, k=10):
    
    if user_id not in user_indices:
        return np.nan
    
    if movie_id not in mean_centered_matrix.columns:
        return np.nan
    
    user_index = user_indices[user_id]
    
    movie_centered_ratings = mean_centered_matrix[movie_id]
    
    rated_users = movie_centered_ratings.notna()
    
    if not rated_users.any():
        return np.nan
    
    rated_user_ids = movie_centered_ratings[rated_users].index
    
    rated_user_indices = [
        user_indices[u]
        for u in rated_user_ids
    ]
    
    similarities = centered_user_similarity[
        user_index
    ][rated_user_indices]
    
    centered_ratings = movie_centered_ratings[
        rated_users
    ].values
    
    top_indices = np.argsort(similarities)[::-1][:k]
    
    top_similarities = similarities[top_indices]
    top_centered_ratings = centered_ratings[top_indices]
    
    similarity_sum = np.abs(top_similarities).sum()
    
    if similarity_sum == 0:
        return np.nan
    
    weighted_deviation = (
        np.sum(
            top_similarities * top_centered_ratings
        ) / similarity_sum
    )
    
    predicted_rating = (
        user_mean_ratings[user_id] +
        weighted_deviation
    )
    
    return np.clip(predicted_rating, 1, 5)

In [30]:
# Test the mean-centered prediction on one unseen rating.

sample = test_data.iloc[0]

centered_prediction = predict_centered_rating(
    sample["user_id"],
    sample["movie_id"]
)

print("User ID:", sample["user_id"])
print("Movie ID:", sample["movie_id"])
print("Actual rating:", sample["rating"])
print("Centered prediction:", centered_prediction)

User ID: 5412
Movie ID: 2683
Actual rating: 2
Centered prediction: 3.613300040178333


In [31]:
# Generate predictions for the same evaluation sample.

centered_predictions = []
centered_actual_ratings = []

for _, row in evaluation_sample.iterrows():
    
    prediction = predict_centered_rating(
        row["user_id"],
        row["movie_id"]
    )
    
    if not np.isnan(prediction):
        centered_predictions.append(prediction)
        centered_actual_ratings.append(row["rating"])

print("Predictions generated:", len(centered_predictions))

Predictions generated: 5000


In [32]:
# Calculate the RMSE of the mean-centered collaborative filtering model.

centered_rmse = np.sqrt(
    mean_squared_error(
        centered_actual_ratings,
        centered_predictions
    )
)

print("Mean-centered CF RMSE:", centered_rmse)

Mean-centered CF RMSE: 0.9062351816687647


In [33]:
# Compare all rating prediction models.

model_comparison = pd.DataFrame({
    "Model": [
        "Baseline - Average Rating",
        "User-Based Collaborative Filtering",
        "Mean-Centered User-Based Collaborative Filtering"
    ],
    "RMSE": [
        baseline_rmse,
        rmse,
        centered_rmse
    ]
})

model_comparison.sort_values("RMSE")

,Model,RMSE
2,Mean-Centered User-Based Collaborative Filtering,0.906235
1,User-Based Collaborative Filtering,0.983540
0,Baseline - Average Rating,1.117357


## Matrix Factorization using SVD

Matrix Factorization represents users and movies using latent factors.
These factors are used to estimate how much a user may like a movie.

In [46]:
# Import the SVD algorithm for matrix factorization.

from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

In [47]:
# Prepare the training matrix for SVD.

svd_matrix = user_movie_matrix.fillna(0)

print("Matrix shape:", svd_matrix.shape)

Matrix shape: (6040, 3683)


In [48]:
# Convert the user-movie matrix into a sparse matrix.

svd_sparse_matrix = csr_matrix(svd_matrix.values)

print("Sparse matrix shape:", svd_sparse_matrix.shape)
print("Non-zero ratings:", svd_sparse_matrix.nnz)

Sparse matrix shape: (6040, 3683)
Non-zero ratings: 800167


In [49]:
# Create the SVD model with 50 latent factors.

svd_model = TruncatedSVD(
    n_components=50,
    random_state=42
)

print(svd_model)

TruncatedSVD(n_components=50, random_state=42)


In [50]:
# Learn the latent factors from the training matrix.

user_latent_matrix = svd_model.fit_transform(
    svd_sparse_matrix
)

print("Original matrix shape:", svd_sparse_matrix.shape)
print("Latent matrix shape:", user_latent_matrix.shape)

Original matrix shape: (6040, 3683)
Latent matrix shape: (6040, 50)


In [51]:
# Calculate the total variance captured by the selected latent factors.

explained_variance = svd_model.explained_variance_ratio_.sum()

print("Explained variance:", explained_variance)

Explained variance: 0.33959781035415404


In [52]:
# Extract the latent representation of each movie.

movie_latent_matrix = svd_model.components_.T

print("Movie latent matrix shape:", movie_latent_matrix.shape)

Movie latent matrix shape: (3683, 50)


In [53]:
# Create a mapping from movie IDs to their positions in the SVD matrix.

movie_indices = {
    movie_id: index
    for index, movie_id in enumerate(user_movie_matrix.columns)
}

print("Number of movies:", len(movie_indices))

Number of movies: 3683


In [54]:
# Predict a user's rating for a movie using the SVD model.

def predict_svd_rating(user_id, movie_id):
    
    if user_id not in user_indices:
        return np.nan
    
    if movie_id not in movie_indices:
        return np.nan
    
    user_index = user_indices[user_id]
    movie_index = movie_indices[movie_id]
    
    user_vector = user_latent_matrix[user_index]
    movie_vector = movie_latent_matrix[movie_index]
    
    predicted_rating = np.dot(
        user_vector,
        movie_vector
    )
    
    return np.clip(predicted_rating, 1, 5)

In [55]:
# Test the SVD rating prediction on one unseen rating.

sample = test_data.iloc[0]

svd_prediction = predict_svd_rating(
    sample["user_id"],
    sample["movie_id"]
)

print("User ID:", sample["user_id"])
print("Movie ID:", sample["movie_id"])
print("Actual rating:", sample["rating"])
print("SVD prediction:", svd_prediction)

User ID: 5412
Movie ID: 2683
Actual rating: 2
SVD prediction: 1.4295369413750494


In [56]:
# Generate SVD predictions for the evaluation sample.

svd_predictions = []
svd_actual_ratings = []

for _, row in evaluation_sample.iterrows():
    
    prediction = predict_svd_rating(
        row["user_id"],
        row["movie_id"]
    )
    
    if not np.isnan(prediction):
        svd_predictions.append(prediction)
        svd_actual_ratings.append(row["rating"])

print("Predictions generated:", len(svd_predictions))

Predictions generated: 5000


In [57]:
# Calculate the RMSE of the SVD model.

svd_rmse = np.sqrt(
    mean_squared_error(
        svd_actual_ratings,
        svd_predictions
    )
)

print("SVD RMSE:", svd_rmse)

SVD RMSE: 2.4511048858068305


In [58]:
# Compare the SVD model with the previous collaborative filtering models.

final_model_comparison = pd.DataFrame({
    "Model": [
        "Baseline - Average Rating",
        "User-Based Collaborative Filtering",
        "Mean-Centered User-Based Collaborative Filtering",
        "SVD Matrix Factorization"
    ],
    "RMSE": [
        baseline_rmse,
        rmse,
        centered_rmse,
        svd_rmse
    ]
})

final_model_comparison.sort_values(
    "RMSE"
)

,Model,RMSE
2,Mean-Centered User-Based Collaborative Filtering,0.906235
1,User-Based Collaborative Filtering,0.983540
0,Baseline - Average Rating,1.117357
3,SVD Matrix Factorization,2.451105


In [59]:
# Select a user for generating movie recommendations.

target_user_id = 1

print("Target user:", target_user_id)

Target user: 1


In [60]:
# Find the most similar users to the target user.

target_user_index = user_indices[target_user_id]

similarity_scores = centered_user_similarity[target_user_index]

similar_user_indices = (
    np.argsort(similarity_scores)[::-1]
)

similar_user_indices = [
    index
    for index in similar_user_indices
    if index != target_user_index
    and similarity_scores[index] > 0
][:50]

similar_user_ids = user_movie_matrix.index[
    similar_user_indices
]

similar_user_scores = similarity_scores[
    similar_user_indices
]

similar_users_df = pd.DataFrame({
    "user_id": similar_user_ids,
    "similarity": similar_user_scores
})

similar_users_df.head(10)

,user_id,similarity
0,1337,0.254381
1,2495,0.191836
2,49,0.181834
3,4724,0.179879
4,5635,0.179316
5,379,0.179045
6,2459,0.164729
7,1515,0.164223
8,942,0.148980
9,3784,0.147941


In [61]:
# Get the ratings given by the similar users.

similar_user_ratings = mean_centered_matrix.loc[
    similar_user_ids
]

print("Similar users:", similar_user_ratings.shape[0])
print("Movies:", similar_user_ratings.shape[1])

Similar users: 50
Movies: 3683


In [67]:
# Calculate similarity-weighted preference scores for each movie.

weights = pd.Series(
    similar_user_scores,
    index=similar_user_ids
)

weighted_sum = (
    similar_user_ratings
    .mul(weights, axis=0)
    .sum(axis=0)
)

similarity_sum = (
    similar_user_ratings
    .notna()
    .mul(weights, axis=0)
    .sum(axis=0)
)

rating_counts = (
    similar_user_ratings
    .notna()
    .sum(axis=0)
)

recommendation_scores = pd.DataFrame({
    "weighted_sum": weighted_sum,
    "similarity_sum": similarity_sum,
    "rating_count": rating_counts
})

recommendation_scores["predicted_rating"] = (
    user_mean_ratings[target_user_id]
    + (
        recommendation_scores["weighted_sum"]
        / recommendation_scores["similarity_sum"].replace(0, np.nan)
    )
)

recommendation_scores["predicted_rating"] = (
    recommendation_scores["predicted_rating"]
    .clip(1, 5)
)

In [68]:
# Remove movies that the target user has already rated.

user_rated_movies = set(
    train_data[
        train_data["user_id"] == target_user_id
    ]["movie_id"]
)

recommendation_scores = recommendation_scores[
    ~recommendation_scores.index.isin(user_rated_movies)
]

print(
    "Candidate movies:",
    len(recommendation_scores)
)

Candidate movies: 3642


In [69]:
# Keep movies that have enough ratings from similar users.

recommendation_scores = recommendation_scores[
    recommendation_scores["rating_count"] >= 5
]

recommendation_scores = recommendation_scores[
    (recommendation_scores["predicted_rating"] >= 1) &
    (recommendation_scores["predicted_rating"] <= 5)
]

print(
    "Reliable candidate movies:",
    len(recommendation_scores)
)

Reliable candidate movies: 151


In [70]:
# Rank the candidate movies and select the top 10 recommendations.

top_recommendations = (
    recommendation_scores
    .sort_values(
        "predicted_rating",
        ascending=False
    )
    .head(10)
    .reset_index()
    .rename(columns={"index": "movie_id"})
)

top_recommendations = top_recommendations.merge(
    movies[["movie_id", "title", "genres"]],
    on="movie_id",
    how="left"
)

top_recommendations[
    [
        "movie_id",
        "title",
        "genres",
        "predicted_rating",
        "rating_count"
    ]
]

,movie_id,title,genres,predicted_rating,rating_count
0,3035,Mister Roberts (1955),Comedy|Drama|War,5.000000,5
1,3753,"Patriot, The (2000)",Action|Drama|War,5.000000,6
2,1278,Young Frankenstein (1974),Comedy|Horror,5.000000,8
3,597,Pretty Woman (1990),Comedy|Romance,4.911782,6
4,1610,"Hunt for Red October, The (1990)",Action|Thriller,4.910575,7
5,3363,American Graffiti (1973),Comedy|Drama,4.893624,6
6,1947,West Side Story (1961),Musical|Romance,4.868686,8
7,2087,Peter Pan (1953),Animation|Children's|Fantasy|Musical,4.864460,5
8,1358,Sling Blade (1996),Drama|Thriller,4.863718,5
9,1200,Aliens (1986),Action|Sci-Fi|Thriller|War,4.863356,8


In [71]:
# Create a reusable function to generate personalized movie recommendations.

def recommend_movies(user_id, n=10):
    
    if user_id not in user_indices:
        print("User ID not found.")
        return
    
    # Find the user's position in the similarity matrix
    target_user_index = user_indices[user_id]
    
    # Get similarity scores with all users
    similarity_scores = centered_user_similarity[target_user_index]
    
    # Find the 50 most similar users
    similar_user_indices = np.argsort(
        similarity_scores
    )[::-1]
    
    similar_user_indices = [
        index
        for index in similar_user_indices
        if index != target_user_index
        and similarity_scores[index] > 0
    ][:50]
    
    similar_user_ids = user_movie_matrix.index[
        similar_user_indices
    ]
    
    similar_user_scores = similarity_scores[
        similar_user_indices
    ]
    
    # Get ratings of similar users
    similar_user_ratings = mean_centered_matrix.loc[
        similar_user_ids
    ]
    
    # Convert similarities into a Series
    weights = pd.Series(
        similar_user_scores,
        index=similar_user_ids
    )
    
    # Calculate weighted rating deviations
    weighted_sum = (
        similar_user_ratings
        .mul(weights, axis=0)
        .sum(axis=0)
    )
    
    # Calculate the sum of similarities for users
    # who actually rated each movie
    similarity_sum = (
        similar_user_ratings
        .notna()
        .mul(weights, axis=0)
        .sum(axis=0)
    )
    
    # Count how many similar users rated each movie
    rating_counts = (
        similar_user_ratings
        .notna()
        .sum(axis=0)
    )
    
    # Create recommendation dataframe
    recommendation_scores = pd.DataFrame({
        "weighted_sum": weighted_sum,
        "similarity_sum": similarity_sum,
        "rating_count": rating_counts
    })
    
    # Calculate predicted rating
    recommendation_scores["predicted_rating"] = (
        user_mean_ratings[user_id]
        + (
            recommendation_scores["weighted_sum"]
            / recommendation_scores[
                "similarity_sum"
            ].replace(0, np.nan)
        )
    )
    
    # Keep predictions within the rating range
    recommendation_scores["predicted_rating"] = (
        recommendation_scores["predicted_rating"]
        .clip(1, 5)
    )
    
    # Remove movies already rated by the user
    user_rated_movies = set(
        train_data[
            train_data["user_id"] == user_id
        ]["movie_id"]
    )
    
    recommendation_scores = recommendation_scores[
        ~recommendation_scores.index.isin(
            user_rated_movies
        )
    ]
    
    # Keep movies with enough supporting ratings
    recommendation_scores = recommendation_scores[
        recommendation_scores["rating_count"] >= 5
    ]
    
    # Select top N recommendations
    top_recommendations = (
        recommendation_scores
        .sort_values(
            "predicted_rating",
            ascending=False
        )
        .head(n)
        .reset_index()
        .rename(columns={"index": "movie_id"})
    )
    
    # Add movie information
    top_recommendations = top_recommendations.merge(
        movies[["movie_id", "title", "genres"]],
        on="movie_id",
        how="left"
    )
    
    return top_recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "predicted_rating",
            "rating_count"
        ]
    ]

In [72]:
# Generate recommendations for User 1 using the reusable function.

recommend_movies(1)

,movie_id,title,genres,predicted_rating,rating_count
0,3035,Mister Roberts (1955),Comedy|Drama|War,5.000000,5
1,3753,"Patriot, The (2000)",Action|Drama|War,5.000000,6
2,1278,Young Frankenstein (1974),Comedy|Horror,5.000000,8
3,597,Pretty Woman (1990),Comedy|Romance,4.911782,6
4,1610,"Hunt for Red October, The (1990)",Action|Thriller,4.910575,7
5,3363,American Graffiti (1973),Comedy|Drama,4.893624,6
6,1947,West Side Story (1961),Musical|Romance,4.868686,8
7,2087,Peter Pan (1953),Animation|Children's|Fantasy|Musical,4.864460,5
8,1358,Sling Blade (1996),Drama|Thriller,4.863718,5
9,1200,Aliens (1986),Action|Sci-Fi|Thriller|War,4.863356,8


In [73]:
# Generate recommendations for another user.

recommend_movies(100)

,movie_id,title,genres,predicted_rating,rating_count
0,50,"Usual Suspects, The (1995)",Crime|Thriller,4.060099,14
1,1278,Young Frankenstein (1974),Comedy|Horror,3.943933,10
2,2706,American Pie (1999),Comedy,3.923193,5
3,912,Casablanca (1942),Drama|Romance|War,3.910875,10
4,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,3.909791,28
5,1089,Reservoir Dogs (1992),Crime|Thriller,3.907809,6
6,2762,"Sixth Sense, The (1999)",Thriller,3.905197,14
7,913,"Maltese Falcon, The (1941)",Film-Noir|Mystery,3.900980,6
8,1307,When Harry Met Sally... (1989),Comedy|Romance,3.872218,7
9,1136,Monty Python and the Holy Grail (1974),Comedy,3.863677,10


In [75]:
# Create a better recommendation function with separate ranking and display scores.

def recommend_movies(user_id, n=10):
    
    if user_id not in user_indices:
        print("User ID not found.")
        return
    
    # Find the target user's position
    target_user_index = user_indices[user_id]
    
    # Get similarity scores
    similarity_scores = centered_user_similarity[target_user_index]
    
    # Find the 50 most similar users
    similar_user_indices = np.argsort(
        similarity_scores
    )[::-1]
    
    similar_user_indices = [
        index
        for index in similar_user_indices
        if index != target_user_index
        and similarity_scores[index] > 0
    ][:50]
    
    similar_user_ids = user_movie_matrix.index[
        similar_user_indices
    ]
    
    similar_user_scores = similarity_scores[
        similar_user_indices
    ]
    
    # Get mean-centered ratings
    similar_user_ratings = mean_centered_matrix.loc[
        similar_user_ids
    ]
    
    # Convert similarities into weights
    weights = pd.Series(
        similar_user_scores,
        index=similar_user_ids
    )
    
    # Calculate weighted rating deviations
    weighted_sum = (
        similar_user_ratings
        .mul(weights, axis=0)
        .sum(axis=0)
    )
    
    # Calculate similarity of users who rated each movie
    similarity_sum = (
        similar_user_ratings
        .notna()
        .mul(weights, axis=0)
        .sum(axis=0)
    )
    
    # Count supporting ratings
    rating_counts = (
        similar_user_ratings
        .notna()
        .sum(axis=0)
    )
    
    recommendation_scores = pd.DataFrame({
        "weighted_sum": weighted_sum,
        "similarity_sum": similarity_sum,
        "rating_count": rating_counts
    })
    
    # Calculate the predicted rating
    recommendation_scores["predicted_rating"] = (
        user_mean_ratings[user_id]
        + (
            recommendation_scores["weighted_sum"]
            / recommendation_scores["similarity_sum"].replace(
                0, np.nan
            )
        )
    )
    
    # Keep the raw prediction for ranking
    recommendation_scores["ranking_score"] = (
        recommendation_scores["predicted_rating"]
    )
    
    # Limit displayed rating to the valid 1–5 range
    recommendation_scores["predicted_rating"] = (
        recommendation_scores["predicted_rating"]
        .clip(1, 5)
    )
    
    # Remove movies already rated by the user
    user_rated_movies = set(
        train_data[
            train_data["user_id"] == user_id
        ]["movie_id"]
    )
    
    recommendation_scores = recommendation_scores[
        ~recommendation_scores.index.isin(
            user_rated_movies
        )
    ]
    
    # Require enough supporting ratings
    recommendation_scores = recommendation_scores[
        recommendation_scores["rating_count"] >= 5
    ]
    
    # Rank using the raw score
    top_recommendations = (
        recommendation_scores
        .sort_values(
            "ranking_score",
            ascending=False
        )
        .head(n)
        .reset_index()
        .rename(columns={"index": "movie_id"})
    )
    
    # Add movie information
    top_recommendations = top_recommendations.merge(
        movies[["movie_id", "title", "genres"]],
        on="movie_id",
        how="left"
    )
    
    return top_recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "predicted_rating",
            "rating_count"
        ]
    ]

In [76]:
# Generate the improved recommendations for User 1.

recommend_movies(1)

,movie_id,title,genres,predicted_rating,rating_count
0,3753,"Patriot, The (2000)",Action|Drama|War,5.000000,6
1,3035,Mister Roberts (1955),Comedy|Drama|War,5.000000,5
2,1278,Young Frankenstein (1974),Comedy|Horror,5.000000,8
3,597,Pretty Woman (1990),Comedy|Romance,4.911782,6
4,1610,"Hunt for Red October, The (1990)",Action|Thriller,4.910575,7
5,3363,American Graffiti (1973),Comedy|Drama,4.893624,6
6,1947,West Side Story (1961),Musical|Romance,4.868686,8
7,2087,Peter Pan (1953),Animation|Children's|Fantasy|Musical,4.864460,5
8,1358,Sling Blade (1996),Drama|Thriller,4.863718,5
9,1200,Aliens (1986),Action|Sci-Fi|Thriller|War,4.863356,8


In [77]:
# Generate the improved recommendations for User 100.

recommend_movies(100)

,movie_id,title,genres,predicted_rating,rating_count
0,50,"Usual Suspects, The (1995)",Crime|Thriller,4.060099,14
1,1278,Young Frankenstein (1974),Comedy|Horror,3.943933,10
2,2706,American Pie (1999),Comedy,3.923193,5
3,912,Casablanca (1942),Drama|Romance|War,3.910875,10
4,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,3.909791,28
5,1089,Reservoir Dogs (1992),Crime|Thriller,3.907809,6
6,2762,"Sixth Sense, The (1999)",Thriller,3.905197,14
7,913,"Maltese Falcon, The (1941)",Film-Noir|Mystery,3.900980,6
8,1307,When Harry Met Sally... (1989),Comedy|Romance,3.872218,7
9,1136,Monty Python and the Holy Grail (1974),Comedy,3.863677,10
